# VarDyn-QG WHIRLS run
This notebook runs the VarDyn-QG configuration for the WHIRLS Agulhas experiment, from setup through assimilation and diagnostics.


Select the GPU device that JAX/CUDA should use for this notebook session.


In [ ]:
import os 
NUM_GPU = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = NUM_GPU

Point the notebook to the WHIRLS VarDyn-QG configuration file.


In [2]:
path_config = 'config_VarDyn-QG.py'

Add the local VarDyn mapping package to sys.path so the src modules can be imported from the notebook.


In [3]:
import sys
sys.path.append('/home/flo/VarDyn/mapping')

Load and merge the experiment configuration with the VarDyn defaults.


In [ ]:
from src import exp
config = exp.Exp(path_config)

Build the model grid and state object, then display the land/sea mask as a quick geometry check.


In [ ]:
from src import state as state
State = state.State(config)

In [ ]:
import matplotlib.pylab as plt
plt.pcolormesh(State.lon, State.lat,State.mask)
plt.colorbar()

Instantiate the dynamical model selected by the configuration.


In [6]:
from src import mod as mod
Model = mod.Model(config,State)

Open, filter, and preprocess the observation datasets defined in the configuration.


In [7]:
from src import obs as obs
dict_obs = obs.Obs(config,State)

Build the observation operators that interpolate model variables to the observation locations or grids.


In [8]:
from src import obsop as obsop
Obsop = obsop.Obsop(config,State,dict_obs,Model)

Construct the reduced basis used as the 4DVar control-vector representation.


In [9]:
from src import basis as basis
Basis = basis.Basis(config,State)

Run the 4DVar inversion with the configured model, observations, operators, and basis.


In [ ]:
from src import inv as inv
inv.Inv(config,State,Model,dict_obs=dict_obs,Obsop=Obsop,Basis=Basis,gpu_device=NUM_GPU)  